#### Structured Output

In [1]:
import os 
from langchain.chat_models import init_chat_model 
os.environ['GROQ_API_KEY']=os.getenv('GROQ_API_KEY')


model=init_chat_model('groq:qwen/qwen3-32b')
model

ChatGroq(output_version=None, profile={'max_input_tokens': 131072, 'max_output_tokens': 16384, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True}, client=<groq.resources.chat.completions.Completions object at 0x10e309940>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x10e30a660>, model_name='qwen/qwen3-32b', model_kwargs={}, groq_api_key=SecretStr('**********'), groq_api_base=None, groq_proxy=None)

In [4]:
from pydantic import BaseModel,Field

class Movie(BaseModel):
    title:str=Field(description="the title of the movie")
    year:int=Field(description="this year the movie was realesed")
    director:str=Field(description="the director of the movie")
    rating:float=Field(description="the movies rating out of 10")


In [6]:
model_with_structure=model.with_structured_output(Movie)

model_with_structure

_ChatModelBinding(bound=ChatGroq(output_version=None, profile={'max_input_tokens': 131072, 'max_output_tokens': 16384, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True}, client=<groq.resources.chat.completions.Completions object at 0x10e309940>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x10e30a660>, model_name='qwen/qwen3-32b', model_kwargs={}, groq_api_key=SecretStr('**********'), groq_api_base=None, groq_proxy=None), kwargs={'tools': [{'type': 'function', 'function': {'name': 'Movie', 'description': '', 'parameters': {'properties': {'title': {'description': 'the title of the movie', 'type': 'string'}, 'year': {'description': 'this year the movie was realesed', 'type': 'integer'}, 'director': {'description': 'the director of the movie', 'type': 'string'}, 'rating': {'description': 'the movies rating out of 10', 'type

In [10]:
model.invoke("provide details about the movie Inception")

AIMessage(content='<think>\nOkay, I need to provide details about the movie Inception. Let me start by recalling what I know. It\'s a 2010 movie directed by Christopher Nolan. The main actor is Leonardo DiCaprio. The title itself is "Inception," which I think means the act of starting something. The movie is about dreams and entering people\'s dreams to plant ideas. \n\nThe main character, Dom Cobb, is a thief who steals information from people\'s subconscious by entering their dreams. That\'s called extraction. He\'s offered a chance to have his criminal history erased by performing the reverse: planting an idea in someone\'s mind, which is called inception. So the main plot is about executing this inception.\n\nThe cast includes some big names. Leonardo DiCaprio as Cobb, Joseph Gordon-Levitt as Arthur, the team\'s operations man. Ellen Page (now Eryn Krueger) plays Ariadne, the architect who designs the dream environments. Tom Hardy is the antagonist, who\'s a rival thief. There\'s a

In [13]:
response=model_with_structure.invoke("provide details about the movie Inception")
print(response)

title='Inception' year=2010 director='Christopher Nolan' rating=8.8


#### Message Output alongside parsed structure 

In [16]:
from pydantic import BaseModel,Field


class Movie(BaseModel):
    """A movie with details""" 
    title:str=Field(...,description="the title of the movie")
    year:int=Field(...,description="this year the movie was realesed")
    director:str=Field(...,description="the director of the movie")
    rating:float=Field(...,description="the movies rating out of 10")

model_with_structure=model.with_structured_output(Movie,include_raw=True)
    

response=model_with_structure.invoke("Provide details about the movie Inception")
response

{'raw': AIMessage(content='', additional_kwargs={'reasoning_content': 'Okay, the user is asking for details about the movie Inception. Let me check what tools I have available. There\'s a Movie function that requires title, year, director, and rating. I need to make sure I have all that information for Inception. \n\nFirst, the title is obviously "Inception". The year it was released was 2010. The director is Christopher Nolan. As for the rating, I think it\'s around 8.8 on IMDb. Let me confirm that. Yes, IMDb does give it an 8.8. So I have all the required parameters. I\'ll structure the tool call with these details. Make sure the JSON is correctly formatted with the right data types: year as integer, rating as a number. No typos in the director\'s name. Alright, that should cover it.\n', 'tool_calls': [{'id': 'ab3yxn1cz', 'function': {'arguments': '{"director":"Christopher Nolan","rating":8.8,"title":"Inception","year":2010}', 'name': 'Movie'}, 'type': 'function'}]}, response_metadat